# 02_pipeline — governed pipeline output orchestration template

Read source data, register DataFrames, profile data, transform into pipeline outputs, optionally enrich metadata and author guardrails through widgets, enforce guardrails, then write pipeline outputs and runtime metadata.

Only edit the source reads, transformations, output write settings, and lineage relationships. Schema, freshness, profile, DQ guardrail authoring, and enrichment are handled by widgets after profiles exist.


## 1. Run `00_env_config`


In [ ]:
%run 00_env_config


In [ ]:
# ============================================================
# Notebook display settings
# Usually no change needed
# ============================================================

# display_guardrail_results defaults to summary mode. Set this only when you need detailed/debug output.
GUARDRAIL_DISPLAY_MODE = "summary"


## 2. Import required functions


In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    display_guardrail_results,
    prepare_pipeline_table_configs,
    profile_dataframe,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    run_table_guardrails,
    widget_pipeline_bootstrap,
    widget_author_dq_rules,
    widget_author_schema_freshness_profile_rules,
    widget_enrich_table_metadata,
    widget_review_guardrail_governance,
    widget_select_guardrail_target,
    write_lakehouse_table,
    write_warehouse_table,
    write_pipeline_lineage,
    write_pipeline_run_summary,
)


## 3. Select agreement and capture run context


In [ ]:
PIPELINE = widget_pipeline_bootstrap(
    notebook_type="02_pipeline",
    select_agreement=True,
    register_notebook=True,
)


---
# SOURCE AREA

Read source data, register key + DataFrame only, then profile data.


## Optional bootstrap from Warehouse to Source Lakehouse Delta

Use this only when the source data currently lives in a Fabric Warehouse but the rest of the pipeline will run in PySpark. FabricOps is optimized for Lakehouse Delta for Spark processing. Warehouse reads from Spark use the Fabric Warehouse connector path, which can have performance impact for large tables, so prefer pushed-down queries and Lakehouse Delta materialization for larger data.

Recommended path: **Warehouse → Source Lakehouse Delta → PySpark transforms → Unified/Product Delta → optional Warehouse publish**.

`read_warehouse_query` is preferred for large Warehouse sources because filters/projections run before Spark receives data.

`read_warehouse_table` reads the full `schema.table` and should be used only for small tables, lookup tables, smoke tests, or intentional full-table reads.

For large or repeated processing, first copy the filtered/projected Warehouse result into the Source Lakehouse as Delta, then continue the pipeline from `read_lakehouse_table(...)`.

Rule of thumb: small means under roughly 1M rows or 1 GB, narrow table, one-time or ad hoc; medium means 1M to 10M rows or 1 to 10 GB, benchmark first; large means over 10M rows, over 10 GB, hundreds of columns, large text fields, or repeated processing, materialize to Lakehouse Delta first; very large should use Fabric Copy/Data Factory style movement or chunked incremental loads, not one notebook cell.


In [ ]:
# ============================================================
# Optional Warehouse to Source Lakehouse Delta bootstrap
# Disabled by default. Configure intentionally before running.
# ============================================================

RUN_WAREHOUSE_TO_LAKEHOUSE_BOOTSTRAP = False

WAREHOUSE_TARGET = "warehouse"
WAREHOUSE_SCHEMA = "dbo"
WAREHOUSE_TABLE = "SOURCE_TABLE"

SOURCE_LAKEHOUSE_TARGET = "source"
SOURCE_LAKEHOUSE_SCHEMA = None
SOURCE_LAKEHOUSE_TABLE = "SOURCE_TABLE"

PARALLEL_LOAD_COLUMN = ""  # required for chunked large table loads
LOAD_MODE = "append"
LOWER_BOUND = None
UPPER_BOUND = None
CHUNK_SIZE = None
OUTPUT_PARTITION_COLUMNS = None

if RUN_WAREHOUSE_TO_LAKEHOUSE_BOOTSTRAP:
    # Example A: small table full copy. Only use this for small reference or ad hoc tables.
    # More generally, reserve read_warehouse_table for small tables, lookup tables,
    # smoke tests, or intentional full-table reads. Prefer read_warehouse_query for larger data.
    # For large tables, use chunked incremental loading or Fabric Copy activity instead.
    if LOAD_MODE == "small_full_copy":
        df_warehouse_source = read_warehouse_table(
            WAREHOUSE_SCHEMA,
            WAREHOUSE_TABLE,
            target=WAREHOUSE_TARGET,
            spark_session=spark,
        )
        write_lakehouse_table(
            df_warehouse_source,
            SOURCE_LAKEHOUSE_TABLE,
            target=SOURCE_LAKEHOUSE_TARGET,
            schema=SOURCE_LAKEHOUSE_SCHEMA,
            mode="overwrite",
            partition_by=OUTPUT_PARTITION_COLUMNS,
            options={"overwriteSchema": "true"},
        )
    else:
        # Example B: chunked Warehouse to Lakehouse bootstrap.
        # Configure a date, integer, identity, or stable partition/watermark column first.
        if not PARALLEL_LOAD_COLUMN:
            raise ValueError(
                "Set PARALLEL_LOAD_COLUMN before running Warehouse to Lakehouse bootstrap. "
                "Use a date, integer, identity, or stable partition column so the load can be chunked."
            )
        if LOWER_BOUND is None or UPPER_BOUND is None or CHUNK_SIZE is None:
            raise ValueError("Set LOWER_BOUND, UPPER_BOUND, and CHUNK_SIZE before running chunked bootstrap.")

        current_lower = LOWER_BOUND
        chunk_number = 0
        while current_lower < UPPER_BOUND:
            current_upper = min(current_lower + CHUNK_SIZE, UPPER_BOUND)
            current_mode = "overwrite" if LOAD_MODE == "overwrite" and chunk_number == 0 else "append"
            query = f"""
SELECT *
FROM {WAREHOUSE_SCHEMA}.{WAREHOUSE_TABLE}
WHERE {PARALLEL_LOAD_COLUMN} >= {current_lower}
  AND {PARALLEL_LOAD_COLUMN} < {current_upper}
"""
            print(f"Loading Warehouse chunk {chunk_number + 1}: {current_lower} <= {PARALLEL_LOAD_COLUMN} < {current_upper}")
            df_chunk = read_warehouse_query(query, target=WAREHOUSE_TARGET, spark_session=spark)
            write_lakehouse_table(
                df_chunk,
                SOURCE_LAKEHOUSE_TABLE,
                target=SOURCE_LAKEHOUSE_TARGET,
                schema=SOURCE_LAKEHOUSE_SCHEMA,
                mode=current_mode,
                partition_by=OUTPUT_PARTITION_COLUMNS,
                repartition_by=OUTPUT_PARTITION_COLUMNS,
                options={"overwriteSchema": "true"} if current_mode == "overwrite" else None,
            )
            current_lower = current_upper
            chunk_number += 1

        print("Warehouse bootstrap complete. Continue pipeline processing with read_lakehouse_table(...), not direct Warehouse reads.")


## 4. USER EDIT SECTION — read source DataFrames

The read call carries the physical source identity. Registration only needs a stable key and the DataFrame object.


In [ ]:
# ============================================================
# User inputs
# Change this section for your use case
# ============================================================
source_table = "demo_src_orders_happy"  # Change this to your primary input table
customer_table = "demo_src_customers_happy"  # Change this to your secondary input table, if needed

df_orders = read_lakehouse_table(source_table, target="source", schema="DemoTest", spark_session=spark)

df_customers = read_lakehouse_table(customer_table, target="source", schema="DemoTest", spark_session=spark)

# Demo defaults: "demo_src_orders_happy" and "demo_src_customers_happy".

# Optional examples — uncomment only when needed.
# These helpers are thin FabricOps target/path wrappers around standard readers:
# - CSV uses Spark CSV reader options.
# - Excel uses pandas.read_excel options, then converts to Spark DataFrame.
# - Parquet uses Spark Parquet reader options.
# - Warehouse query uses the Fabric Warehouse Spark connector with SQL pushdown.
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.

# CSV file:
# df_orders = read_lakehouse_csv(
#     "input/example.csv",
#     target="source",
#     spark_session=spark,
#     header=True,
#     inferSchema=True,
# )

# Excel file:
# df_orders = read_lakehouse_excel(
#     "input/example.xlsx",
#     target="source",
#     sheet_name=0,
#     spark_session=spark,
# )

# Parquet file:
# df_orders = read_lakehouse_parquet(
#     "input/example.parquet",
#     target="source",
#     spark_session=spark,
# )

# Warehouse query:
# df_orders = read_warehouse_query(
#     "SELECT TOP 1000 * FROM dbo.source_table WHERE business_date >= '2026-01-01'",
#     target="warehouse",
#     spark_session=spark,
# )


### Optional Lakehouse table write/read smoke test


In [ ]:
# For illustration purposes, write the current df_orders to a Lakehouse table first,
# then read it back using the FabricOps Lakehouse table reader.
# This is useful when you want to test table write/read without depending on an existing table.

# write_lakehouse_table(
#     df_orders,
#     table_name="smoke_test_orders_df",
#     target="source",
#     mode="overwrite",
# )

# lakehouse_df = read_lakehouse_table(
#     table_name="smoke_test_orders_df",
#     target="source",
#     spark_session=spark,
# )

# display(lakehouse_df)


## 5. USER EDIT SECTION — register source DataFrames only

Do not define schema, freshness, profile behaviour, DQ, classification, enrichment, or distribution settings here. Those are profiled and curated through widgets.


In [ ]:
SOURCE_TABLES = [
    {"key": "orders", "df": df_orders, "table_name": "orders", "layer": "source", "fabric_store_target": "source"},
    {"key": "customers", "df": df_customers, "table_name": "customers", "layer": "source", "fabric_store_target": "source"},
]

SOURCE_TABLES, SOURCE_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    SOURCE_TABLES,
    {},
    table_role="source",
)

df_orders = SOURCE_CONFIG_BY_KEY["orders"]["df"]
df_customers = SOURCE_CONFIG_BY_KEY["customers"]["df"]


## 6. Profile source data

This records source-side profiles in `METADATA_DATA_CATALOGUE`. It is profile generation, not manual schema authoring.

Optional lightweight `profile_dataframe(...)` usage can be enabled while developing when you want to inspect the primary source DataFrame directly.


In [ ]:
RUN_PROFILE = False
source_table_name = source_table

if RUN_PROFILE:
    profile_df = profile_dataframe(
        df_orders,
        table_name=source_table_name,
    )

    profile_df.show(50, truncate=False)

source_profile_results = run_table_guardrails(
    SOURCE_TABLES,
    table_role="source",
    mode="profile",
)

display_guardrail_results(source_profile_results)


---
# TRANSFORMATION AND TARGET AREA


## 7. USER EDIT SECTION — transform source DataFrames into pipeline outputs


In [ ]:
df_orders_enriched = (
    df_orders.alias("o")
    .join(df_customers.alias("c"), on="customer_id", how="left")
    .select(
        "order_id", "customer_id", "customer_name", "customer_segment",
        F.col("country_code").alias("order_country_code"),
        F.col("customer_country_code"), "order_date", "status", "order_amount",
        F.current_timestamp().alias("processed_ts"),
    )
)

df_orders_summary = (
    df_orders_enriched
    .groupBy("order_date", "customer_segment", "status")
    .agg(F.count("order_id").alias("order_count"), F.sum("order_amount").alias("total_order_amount"))
)


## 8. USER EDIT SECTION — register pipeline outputs only

Pipeline outputs can be profiled before physical output tables exist because the DataFrame already exists.


In [ ]:
TARGET_TABLES = [
    {"key": "orders_enriched", "df": df_orders_enriched, "table_name": "orders_enriched", "layer": "product", "fabric_store_target": "product"},
    {"key": "orders_summary", "df": df_orders_summary, "table_name": "orders_summary", "layer": "product", "fabric_store_target": "product"},
]

TARGET_TABLES, TARGET_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    TARGET_TABLES,
    {},
    table_role="target",
    run_id=PIPELINE.run_id,
    pipeline_name=PIPELINE.pipeline_name,
)

df_orders_enriched = TARGET_CONFIG_BY_KEY["orders_enriched"]["df"]
df_orders_summary = TARGET_CONFIG_BY_KEY["orders_summary"]["df"]


## 9. Profile pipeline outputs


In [ ]:
target_profile_results = run_table_guardrails(
    TARGET_TABLES,
    table_role="target",
    mode="profile",
)

display_guardrail_results(target_profile_results)


## 10. Optional enrichment and guardrail widgets

Run only when you need to author or review schema/freshness/profile/DQ rules or enrich metadata from the latest profiles.


In [ ]:
selected_guardrail_target = widget_select_guardrail_target(spark_session=spark)

widget_author_schema_freshness_profile_rules(
    selected_guardrail_target,
    spark_session=spark,
)
widget_author_dq_rules(
    selected_guardrail_target,
    spark_session=spark,
)
widget_enrich_table_metadata(
    selected_guardrail_target,
    spark_session=spark,
)
widget_review_guardrail_governance(
    selected_guardrail_target,
    spark_session=spark,
)


## 11. Guardrail enforcement gate

If any blocking source or pipeline output guardrail fails, the notebook stops here before output write settings and before any output table write.


In [ ]:
source_enforcement_results = run_table_guardrails(
    SOURCE_TABLES,
    table_role="source",
    mode="enforce",
)
display_guardrail_results(source_enforcement_results)
# run_table_guardrails stops the notebook when blocking source guardrails fail.

target_enforcement_results = run_table_guardrails(
    TARGET_TABLES,
    table_role="target",
    mode="enforce",
)
display_guardrail_results(target_enforcement_results)
# run_table_guardrails stops the notebook when blocking target guardrails fail.


## 12. USER EDIT SECTION — configure output write settings

Write settings belong after the guardrail gate. `target_name` and `write_mode` are essential. Layer, schema, options, partitioning, and repartitioning are write concerns only.


In [ ]:
TARGET_WRITE_SETTINGS = {
    "orders_enriched": {
        "target_layer": "unified",
        "target_name": "demo_unified_orders_enriched",
        "schema": "DemoTest",
        "write_mode": "overwrite",
        "options": {"overwriteSchema": "true"},
    },
    "orders_summary": {
        "target_layer": "unified",
        "target_name": "demo_unified_orders_summary",
        "schema": "DemoTest",
        "write_mode": "overwrite",
        "options": {"overwriteSchema": "true"},
        # Optional: "partition_by": ["order_date"],
        # Optional: "repartition_by": ["customer_segment"],
    },
}

for key, write_settings in TARGET_WRITE_SETTINGS.items():
    TARGET_CONFIG_BY_KEY[key].update(write_settings)


## 13. Write pipeline output Lakehouse tables

This section only runs after all blocking guardrails pass.


In [ ]:
target_write_status = {}

for key, target in TARGET_CONFIG_BY_KEY.items():
    write_lakehouse_table(
        target["df"],
        target["target_name"],
        target=target.get("target_layer", "unified"),
        schema=target.get("schema"),
        mode=target.get("write_mode", "overwrite"),
        partition_by=target.get("partition_by"),
        repartition_by=target.get("repartition_by"),
        options=target.get("options"),
    )
    target_write_status[key] = f"written: {target.get('schema')}.{target['target_name']}"

target_write_status


## 14. Optional warehouse write example


In [ ]:
# orders_summary_target = TARGET_CONFIG_BY_KEY["orders_summary"]
# write_warehouse_table(
#     orders_summary_target["df"],
#     schema="dbo",
#     table_name=orders_summary_target["target_name"],
#     target="warehouse",
#     mode=orders_summary_target.get("write_mode", "overwrite"),
# )


## 15. USER EDIT SECTION — lineage relationships


In [ ]:
LINEAGE_RELATIONSHIPS = [
    {"source_key": "orders", "target_key": "orders_enriched", "transformation_type": "join", "transformation_logic": "orders joined to customers on customer_id"},
    {"source_key": "customers", "target_key": "orders_enriched", "transformation_type": "join", "transformation_logic": "customers joined to orders on customer_id"},
    {"source_key": "orders_enriched", "target_key": "orders_summary", "transformation_type": "aggregation", "transformation_logic": "group by order_date, customer_segment, and status"},
]


## 16. Write lineage metadata


In [ ]:
lineage_result = write_pipeline_lineage(
    spark=spark,
    run_id=PIPELINE.run_id,
    source_definitions=SOURCE_CONFIG_BY_KEY,
    target_definitions=TARGET_CONFIG_BY_KEY,
    relationships=LINEAGE_RELATIONSHIPS,
    pipeline_name=PIPELINE.pipeline_name,
    notebook_id=PIPELINE.notebook_id,
    notebook_registry_id=PIPELINE.notebook_registry_id,
    agreement_id=PIPELINE.agreement_id,
    agreement_contract_version=PIPELINE.agreement_contract_version,
)
lineage_result


## 17. Write runtime summary


In [ ]:
runtime_summary_result = write_pipeline_run_summary(
    source_guardrail_results=source_enforcement_results,
    target_guardrail_results=target_enforcement_results,
    target_write_status=target_write_status,
    lineage_result=lineage_result,
)
runtime_summary_result
